# Causal Uplift — Chạy Causal Forest (model còn lại) trên Colab

**5 model baseline đã chạy local rồi.** Notebook này chỉ chạy **model thứ 6 — Causal Forest** (cần RAM lớn nên phải lên Colab), rồi **ghép với 5 CATE baseline** để ra bảng so sánh đủ 6 model + biểu đồ, **lưu hết vào Google Drive**.

### Chuẩn bị (làm 1 lần)
1. **Runtime > Change runtime type > High-RAM** (hoặc A100/L4).
2. Upload 5 file CATE baseline từ máy — trong `output/cate/`:
   `cate_response.npy`, `cate_s_learner.npy`, `cate_t_learner.npy`, `cate_x_learner.npy`, `cate_dr_learner.npy`
   — lên Google Drive vào thư mục **`MyDrive/causal_uplift_results/cate/`**.
   *(Nếu chưa upload, notebook vẫn chạy Causal Forest bình thường; chỉ là bảng so sánh sẽ thiếu 5 model kia.)*
3. **Runtime > Run all**. Chờ ~90 phút (Causal Forest).

> Giữ nguyên `FRAC=0.50, SEED=42, TEST_SIZE=0.30` để tập test khớp chính xác với baseline local (2.096.940 dòng) → CATE ghép được.

In [ ]:
# === Cell 1: Mount Drive + tự tìm baseline CATE (bất kể để trong thư mục con nào) ===
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUT = Path('/content/drive/MyDrive/causal_uplift_results')
(OUT / 'cate').mkdir(parents=True, exist_ok=True)
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
# tìm mọi cate_*.npy ở BẤT KỲ thư mục con nào dưới causal_uplift_results/ (kể cả output/cate/)
found = {p.name: p for p in OUT.rglob('cate_*.npy')}
print('Baseline CATE tìm thấy:', sorted(found) if found else '(chưa có — notebook vẫn chạy Causal Forest)')
print('Kết quả sẽ lưu vào:', OUT)

In [ ]:
# === Cell 2: Cài thư viện + tải dataset (~1-2 phút) ===
!pip -q install econml==0.16.0 lightgbm==4.5.0
!wget -q https://huggingface.co/datasets/criteo/criteo-uplift/resolve/main/criteo-research-uplift-v2.1.csv.gz -O criteo.csv.gz
!ls -lh criteo.csv.gz

In [ ]:
# === Cell 3: SETUP — holdout + Causal Forest + hàm đánh giá (y hệt src/). Chạy 1 lần ===
import time, warnings, numpy as np, pandas as pd
from sklearn.metrics import auc
from sklearn.model_selection import train_test_split

FRAC, SEED, TEST_SIZE = 0.50, 42, 0.30           # PHẢI khớp local
FEATURES = [f"f{i}" for i in range(12)]

def load_holdout():
    t0 = time.time()
    dtype = {f: "float32" for f in FEATURES}
    dtype.update({"treatment": "int8", "conversion": "int8", "visit": "int8", "exposure": "int8"})
    df = pd.read_csv("criteo.csv.gz", dtype=dtype)
    print(f"[load] {len(df):,} dòng, {time.time()-t0:.0f}s")
    rng = np.random.default_rng(SEED); parts = []
    for _, g in df.groupby(["treatment", "conversion"], sort=False):
        n = max(1, int(round(len(g) * FRAC)))
        idx = rng.choice(g.index.values, size=min(n, len(g)), replace=False)
        parts.append(df.loc[idx])
    df = pd.concat(parts, ignore_index=True)
    strata = df["treatment"].astype(str) + "_" + df["conversion"].astype(str)
    tr, te = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, shuffle=True, stratify=strata)
    tr, te = tr.reset_index(drop=True), te.reset_index(drop=True)
    print(f"[holdout] train={len(tr):,} test={len(te):,}")
    return tr, te

def xty(d):
    return (d[FEATURES].to_numpy("float64"), d["treatment"].to_numpy("float64"), d["conversion"].to_numpy("float64"))

# ---- đánh giá (y hệt src/evaluation.py) ----
def _qini_raw(y, u, t):
    o = np.argsort(u, kind="mergesort")[::-1]; y, t, u = y[o], t[o], u[o]
    yc = np.where(t == 0, y, 0.0); yt = np.where(t == 1, y, 0.0)
    thr = np.r_[np.where(np.diff(u))[0], u.size - 1]
    n1 = np.cumsum(t)[thr]; y1 = np.cumsum(yt)[thr]; n_all = thr + 1; n0 = n_all - n1; y0 = np.cumsum(yc)[thr]
    q = y1 - y0 * np.divide(n1, n0, out=np.zeros_like(n1), where=n0 != 0)
    if n_all.size == 0 or q[0] != 0 or n_all[0] != 0: n_all = np.r_[0, n_all]; q = np.r_[0, q]
    return n_all, q
def _perfect_qini(y, t): return _qini_raw(y, y * t - y * (1 - t), t)
def qini_curve(y, t, u):
    y, t, u = np.asarray(y,"float64"), np.asarray(t,"float64"), np.asarray(u,"float64")
    n, q = _qini_raw(y, u, t); return pd.DataFrame({"n_targeted": n, "qini": q})
def _uplift_raw(y, u, t):
    o = np.argsort(u, kind="mergesort")[::-1]; y, t, u = y[o], t[o], u[o]
    yc = np.where(t == 0, y, 0.0); yt = np.where(t == 1, y, 0.0)
    thr = np.r_[np.where(np.diff(u))[0], u.size - 1]
    n1 = np.cumsum(t)[thr]; y1 = np.cumsum(yt)[thr]; n_all = thr + 1; n0 = n_all - n1; y0 = np.cumsum(yc)[thr]
    r1 = np.divide(y1, n1, out=np.zeros_like(y1), where=n1 != 0); r0 = np.divide(y0, n0, out=np.zeros_like(y0), where=n0 != 0)
    c = (r1 - r0) * n_all
    if n_all.size == 0 or c[0] != 0 or n_all[0] != 0: n_all = np.r_[0, n_all]; c = np.r_[0, c]
    return n_all, c
def _perfect_uplift(y, t):
    cr = np.sum((y == 1) & (t == 0)); tn = np.sum((y == 0) & (t == 1)); s = y if cr > tn else t
    return _uplift_raw(y, 2 * (y == t).astype("float64") + s, t)
def _score(af, pf, y, t, u):
    xa, ya = af(y, u, t); xp, yp = pf(y, t)
    ab = auc(np.array([0, xp[-1]]), np.array([0, yp[-1]])); ap = auc(xp, yp) - ab; aa = auc(xa, ya) - ab
    return float("nan") if ap == 0 else aa / ap
def qini_score(y, t, u):
    y, t, u = np.asarray(y,"float64"), np.asarray(t,"float64"), np.asarray(u,"float64"); return _score(_qini_raw, _perfect_qini, y, t, u)
def auuc_score(y, t, u):
    y, t, u = np.asarray(y,"float64"), np.asarray(t,"float64"), np.asarray(u,"float64"); return _score(_uplift_raw, _perfect_uplift, y, t, u)
def bootstrap_ci(y, t, u, n_boot=500, seed=42, alpha=0.05):
    y, t, u = np.asarray(y,"float64"), np.asarray(t,"float64"), np.asarray(u,"float64")
    n = len(y); rng = np.random.default_rng(seed); s = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for _ in range(n_boot):
            i = rng.integers(0, n, n); s.append(qini_score(y[i], t[i], u[i]))
    s = np.asarray(s); s = s[np.isfinite(s)]; return np.percentile(s, 100*alpha/2), np.percentile(s, 100*(1-alpha/2))
def paired_bootstrap_difference_ci(a, b, y, t, n_boot=500, seed=42, alpha=0.05):
    y, t = np.asarray(y,"float64"), np.asarray(t,"float64"); a, b = np.asarray(a,"float64"), np.asarray(b,"float64")
    n = len(y); rng = np.random.default_rng(seed); d = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for _ in range(n_boot):
            i = rng.integers(0, n, n); d.append(qini_score(y[i], t[i], a[i]) - qini_score(y[i], t[i], b[i]))
    d = np.asarray(d); d = d[np.isfinite(d)]
    point = qini_score(y, t, a) - qini_score(y, t, b)
    if len(d) == 0: return point, float('nan'), float('nan')
    return point, *np.percentile(d, [100*alpha/2, 100*(1-alpha/2)])
print("Setup xong.")

In [ ]:
# === Cell 4: Rebuild holdout + fit CHỈ Causal Forest -> lưu CATE vào Drive (~90 phút) ===
train_df, test_df = load_holdout()
X_tr, T_tr, Y_tr = xty(train_df)
X_te, T_te, Y_te = xty(test_df)
np.savez(OUT / 'cate' / 'holdout_test_yt.npz', Y=Y_te, T=T_te, frac=FRAC, seed=SEED, n_test=len(test_df))

from econml.dml import CausalForestDML
print("[fit] Causal Forest — n_estimators=500, honest=True, cv=3 (ngoại suy ~90 phút @50%)...", flush=True)
t = time.time()
cf = CausalForestDML(n_estimators=500, min_samples_leaf=200, discrete_treatment=True,
                     honest=True, inference=True, cv=3, random_state=SEED)
cf.fit(Y=Y_tr, T=T_tr, X=X_tr)
cate_cf = cf.effect(X_te).ravel()
np.save(OUT / 'cate' / 'cate_causal_forest.npy', cate_cf.astype("float64"))
print(f"[done] Causal Forest {time.time()-t:.0f}s  mean={cate_cf.mean():.6f}  -> Drive", flush=True)

In [ ]:
# === Cell 5: Ghép tất cả CATE có sẵn -> bảng so sánh + phân khúc, lưu Drive ===
MODEL_ORDER = ["Response", "S-Learner", "T-Learner", "X-Learner", "DR-Learner", "Causal Forest"]
def slug(n): return n.lower().replace("-", "_").replace(" ", "_")
# gom mọi cate_*.npy dưới causal_uplift_results/ (bất kể thư mục con), Causal Forest ưu tiên bản vừa tạo
found = {p.name: p for p in OUT.rglob('cate_*.npy')}
cates = {}
for name in MODEL_ORDER:
    fn = f'cate_{slug(name)}.npy'
    if fn in found: cates[name] = np.load(found[fn])
print("Model có CATE:", list(cates))

BASELINE, NBOOT = "T-Learner", 500
rows = []
for name in MODEL_ORDER:
    if name not in cates: continue
    c = cates[name]; q = qini_score(Y_te, T_te, c); a = auuc_score(Y_te, T_te, c)
    lb, ub = bootstrap_ci(Y_te, T_te, c, NBOOT, SEED)
    delta = delta_lb = delta_ub = np.nan
    if name != BASELINE and BASELINE in cates:
        delta, delta_lb, delta_ub = paired_bootstrap_difference_ci(c, cates[BASELINE], Y_te, T_te, NBOOT, SEED)
    rows.append({"model": name, "qini_score": q, "qini_ci_low": lb, "qini_ci_high": ub, "auuc_score": a,
                 "delta_qini_vs_baseline": delta, "delta_qini_ci_low": delta_lb, "delta_qini_ci_high": delta_ub,
                 "sample_frac": FRAC, "n_test": len(Y_te)})
    print(f"[eval] {name:16s} qini={q:.4f} CI=[{lb:.4f},{ub:.4f}] auuc={a:.4f} delta={delta:.4f}")
comp = pd.DataFrame(rows); comp.to_csv(OUT / 'qini_comparison.csv', index=False)
try: display(comp)
except: print(comp)

# Score-sign diagnostic only. Không chọn champion trên holdout test và không diễn giải
# dấu CATE ước lượng như principal strata quan sát được ở cấp cá nhân.
segment_source = "Causal Forest" if "Causal Forest" in cates else next(m for m in MODEL_ORDER if m != "Response" and m in cates)
cb = cates[segment_source]; eps = 1e-4
seg = np.where(cb > eps, "Predicted positive effect", np.where(cb < -eps, "Predicted negative effect", "Near-zero score"))
seg_tbl = (pd.DataFrame({"segment": seg}).assign(cate=cb).groupby("segment")
           .agg(pct_population=("segment", lambda s: round(100*len(s)/len(seg), 2)),
                mean_cate=("cate", lambda s: round(float(s.mean()), 6))).reset_index())
seg_tbl.insert(0, "source_model", segment_source); seg_tbl.to_csv(OUT / 'segments.csv', index=False)
print("Model dùng cho score-sign diagnostic:", segment_source)
try: display(seg_tbl)
except: print(seg_tbl)

In [ ]:
# === Cell 6: Vẽ 4 biểu đồ -> lưu Drive ===
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
PAL = {"Response": "#7c879c", "S-Learner": "#c9a227", "T-Learner": "#b9691f",
       "X-Learner": "#0e7c7b", "DR-Learner": "#5b8def", "Causal Forest": "#b4483a"}
present = [m for m in MODEL_ORDER if m in cates]

plt.figure(figsize=(9, 6))
for m in present:
    cv = qini_curve(Y_te, T_te, cates[m]); plt.plot(cv.n_targeted, cv.qini, label=m, color=PAL[m], lw=2)
anchor = BASELINE if BASELINE in cates else present[-1]
qend = qini_curve(Y_te, T_te, cates[anchor]).qini.iloc[-1]
plt.plot([0, len(Y_te)], [0, qend], "--", color="#999", lw=1, label="Random")
plt.xlabel("Số khách được nhắm (uplift giảm dần)"); plt.ylabel("Qini (conversion tăng thêm)")
plt.title(f"Qini curve — {len(present)} model @ {FRAC:.0%}"); plt.legend(); plt.tight_layout()
plt.savefig(OUT / 'figures' / 'qini_curves.png', dpi=130); plt.show()

plt.figure(figsize=(8, 5)); yv = np.arange(len(comp))[::-1]
plt.errorbar(comp.qini_score, yv, xerr=[comp.qini_score - comp.qini_ci_low, comp.qini_ci_high - comp.qini_score],
             fmt="o", color="#0e7c7b", ecolor="#b9691f", capsize=4, ms=8)
plt.axvline(0, color="#b4483a", ls="--", lw=1); plt.yticks(yv, comp.model)
plt.xlabel("Qini score (± CI 95%)"); plt.title("So sánh model — Qini + CI"); plt.tight_layout()
plt.savefig(OUT / 'figures' / 'model_comparison.png', dpi=130); plt.show()

c = cates[best]; order = np.argsort(c)[::-1]; Ys, Ts = Y_te[order], T_te[order]
n = len(c); edges = np.linspace(0, n, 11, dtype=int); ups = []
for b in range(10):
    lo, hi = edges[b], edges[b+1]; yb, tb = Ys[lo:hi], Ts[lo:hi]
    crt = yb[tb == 1].mean() if (tb == 1).any() else 0; crc = yb[tb == 0].mean() if (tb == 0).any() else 0
    ups.append(crt - crc)
plt.figure(figsize=(9, 5))
plt.bar(range(1, 11), np.array(ups)*100, color=["#0e7c7b" if u >= 0 else "#b4483a" for u in ups])
plt.axhline(0, color="#999", lw=1); plt.xlabel("Decile (1 = uplift cao nhất)"); plt.ylabel("Uplift quan sát (%)")
plt.title(f"Decile lift — {best}"); plt.tight_layout()
plt.savefig(OUT / 'figures' / 'decile_lift.png', dpi=130); plt.show()

plt.figure(figsize=(9, 5))
lo, hi = np.percentile(cates[best], 0.5), np.percentile(cates[best], 99.5)
plt.hist(np.clip(cates[best], lo, hi), bins=60, color="#0e7c7b", alpha=0.85)
plt.axvline(0, color="#b4483a", ls="--", lw=1); plt.xlabel("CATE dự đoán"); plt.ylabel("Số khách")
plt.title(f"Phân bố CATE — {best}"); plt.tight_layout()
plt.savefig(OUT / 'figures' / 'cate_histogram.png', dpi=130); plt.show()

print("Đã lưu 4 biểu đồ + bảng vào:", OUT)